# Orthography Large Experiment

This notebook tracks the expanded orthography experiment as `*_output.jsonl` files land in `batches/orthography_large_exp/`. It mirrors the newer analysis style used in the agreement and error-analysis notebooks: deterministic loaders, progress snapshots that tolerate partial downloads, and compact plotting built around a fixed five-condition orthography palette.

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import re
import sys
import unicodedata
from collections import Counter
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
import pyrootutils
from IPython.display import display

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.precision", 3)

PROJECT_ROOT = pyrootutils.find_root(indicator=".project-root")
DATA_DIR = PROJECT_ROOT / "data"
BATCHES_DIR = PROJECT_ROOT / "batches"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
FIGURES_DIR = PROJECT_ROOT / "paper" / "figures"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

EXP_NAME = "orthography_large"
EXP_DATA_DIR = DATA_DIR / "orthography_large_exp"
EXP_BATCH_DIR = BATCHES_DIR / "orthography_large_exp"
GRAMMAR_LIST = EXP_DATA_DIR / "orthography_large_grammars.txt"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

assert EXP_DATA_DIR.exists(), EXP_DATA_DIR
assert EXP_BATCH_DIR.exists(), EXP_BATCH_DIR
assert GRAMMAR_LIST.exists(), GRAMMAR_LIST

## Define aesthetics

In [ ]:
import aesthetics as aes  # noqa: F401

sns = aes.sns
plt = aes.plt
mtick = aes.mtick

MODEL_ORDER = [
    model
    for model in aes.MODEL_ORDER
    if model.startswith("gpt-5") or "gemma-3" in model
]
ORTHOGRAPHY_ORDER = [
    "Latin → Latin",
    "Latin → Latin (diacritics)",
    "Latin → Cyrillic",
    "Latin → Hebrew",
    "Latin → Hebrew (pointed)",
]
ORTHOGRAPHY_LABELS = {
    "latin": "Latin → Latin",
    "latin_diacritic": "Latin → Latin (diacritics)",
    "cyrillic": "Latin → Cyrillic",
    "hebrew": "Latin → Hebrew (pointed)",
    "hebrew_unpointed": "Latin → Hebrew",
}
PALETTE_ORTHOGRAPHY_LARGE = aes.darken(
    {
        "Latin → Latin": sns.color_palette("Reds", n_colors=7)[1],
        "Latin → Latin (diacritics)": sns.color_palette("Reds", n_colors=7)[3],
        "Latin → Cyrillic": sns.color_palette("Greens", n_colors=7)[2],
        "Latin → Hebrew (pointed)": sns.color_palette("Blues", n_colors=7)[3],
        "Latin → Hebrew": sns.color_palette("Blues", n_colors=7)[5],
    },
    by=0.15,
)
# Set to concrete model names to force plotting that model.
# Leave as None to auto-select from available rows.

METRICS_FOR_PLOT = [
    ("exact_match", "Exact Match"),
    ("bow_match", "Bag of Words"),
    ("bleu", "BLEU Score"),
    ("chrF++", "chrF++"),
]

## Load grammars and samples

In [ ]:
ANSWER_RE = re.compile(
    r"final\s*answer\s*(?::|-|—)?\s*(?:is\s*)?([^\n]+)",
    re.IGNORECASE | re.DOTALL,
)
CYRILLIC_RE = re.compile(r"[а-яА-Я]")
HEBREW_RE = re.compile(r"[\u0590-\u05FF]")


def fuzzy_model(model: str | None) -> str:
    return re.sub(r"-\d{4}-\d{2}-\d{2}$", "", model or "")


def extract_answer(text: str | None) -> str | None:
    if not text:
        return None
    matches = ANSWER_RE.findall(text)
    if not matches:
        return None
    answer = matches[-1].strip()
    answer = re.sub(r"[^\w\s]", "", answer, flags=re.UNICODE).strip()
    return answer or None


def usage_tuple(body: dict) -> tuple[int | None, int | None, int | None]:
    usage = body.get("usage", {}) or {}
    return (
        usage.get("prompt_tokens", usage.get("promptTokens")),
        usage.get("completion_tokens", usage.get("completionTokens")),
        usage.get("total_tokens", usage.get("totalTokens")),
    )


def tokenize(text: str | None) -> list[str]:
    if not isinstance(text, str) or not text.strip():
        return []
    return text.split()


def bag_equal(left: str | None, right: str | None) -> bool:
    return Counter(tokenize(left)) == Counter(tokenize(right))


def detect_script(text: str | None) -> str:
    if not isinstance(text, str) or not text.strip():
        return "empty"
    if CYRILLIC_RE.search(text):
        return "cyrillic"
    if HEBREW_RE.search(text):
        return "hebrew"
    for char in text:
        if char.isalpha() and "LATIN" in unicodedata.name(char, ""):
            return "latin"
    return "other"


def load_grammars(grammar_list: Path) -> pd.DataFrame:
    grammar_ids = [
        line.strip() for line in grammar_list.read_text().splitlines() if line.strip()
    ]
    rows = []
    for grammar_id in grammar_ids:
        grammar = json.loads((EXP_DATA_DIR / f"grammar_{grammar_id}.json").read_text())
        rows.append(
            {
                "grammar_name": grammar_id,
                "n_rules": pd.to_numeric(grammar.get("n_rules"), errors="coerce"),
                "n_words": pd.to_numeric(grammar.get("n_words"), errors="coerce"),
                "grammar_size": pd.to_numeric(grammar.get("n_rules"), errors="coerce")
                + pd.to_numeric(grammar.get("n_words"), errors="coerce"),
                "target_orthography_key": grammar["b"].get("orthography", "unknown"),
                "target_orthography": ORTHOGRAPHY_LABELS.get(
                    grammar["b"].get("orthography", "unknown"),
                    grammar["b"].get("orthography", "unknown"),
                ),
                "syllable_structure": grammar["a"].get("syllable_structure"),
                "target_vocab": sum(
                    [
                        grammar["b"][key]
                        for key in [
                            "verbs",
                            "nouns",
                            "propns",
                            "prons",
                            "adjs",
                            "det_def",
                            "det_indef",
                            "comps",
                        ]
                    ],
                    [],
                ),
            }
        )
    grammar_df = pd.DataFrame(rows)
    grammar_df["target_orthography"] = pd.Categorical(
        grammar_df["target_orthography"],
        categories=ORTHOGRAPHY_ORDER,
        ordered=True,
    )
    return grammar_df


def load_samples(grammar_ids: list[str]) -> pd.DataFrame:
    rows = []
    for grammar_id in grammar_ids:
        sample_path = EXP_DATA_DIR / f"samples_{grammar_id}.jsonl"
        with open(sample_path) as handle:
            for sample_id, line in enumerate(handle):
                sample = json.loads(line)
                rows.append(
                    {
                        "grammar_name": grammar_id,
                        "sample_id": str(sample_id),
                        "input_sentence": sample.get("left_phonetic")
                        or sample.get("left"),
                        "output_sentence": sample.get("right_phonetic")
                        or sample.get("right"),
                        "depth": pd.to_numeric(sample.get("depth"), errors="coerce"),
                    }
                )
    return pd.DataFrame(rows)


grammars_df = load_grammars(GRAMMAR_LIST)
samples_df = load_samples(grammars_df["grammar_name"].tolist()).merge(
    grammars_df[
        ["grammar_name", "target_orthography", "grammar_size", "n_rules", "n_words"]
    ],
    on="grammar_name",
    how="left",
)
samples_df["input_length"] = samples_df["input_sentence"].map(
    lambda text: len(tokenize(text))
)

display(
    samples_df.groupby(["target_orthography", "grammar_size"], observed=False)
    .size()
    .rename("n_samples")
    .reset_index()
)

condition_examples_df = (
    samples_df.sort_values(["target_orthography", "grammar_size", "depth", "sample_id"])
    .groupby("target_orthography", observed=False)
    .head(1)[
        ["target_orthography", "grammar_size", "input_sentence", "output_sentence"]
    ]
    .reset_index(drop=True)
)
condition_examples_df

## Load available batch outputs

In [ ]:
CUSTOM_ID_RE = re.compile(
    r"^(?P<grammar_name>[0-9a-f]+)-(?P<input_hash>[0-9a-f]+)-sample-(?P<sample_id>\d+)$"
)


def load_outputs(batch_dir: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(batch_dir.glob("*_output.jsonl")):
        with open(path) as handle:
            for line in handle:
                item = json.loads(line)
                body = (item.get("response") or {}).get("body") or {}
                choices = body.get("choices") or []
                message = (
                    ((choices[0] or {}).get("message") or {}).get("content")
                    if choices
                    else None
                )
                match = CUSTOM_ID_RE.match(item.get("custom_id", ""))
                if not match:
                    continue
                prompt_tokens, completion_tokens, total_tokens = usage_tuple(body)
                rows.append(
                    {
                        "batch_file": path.name,
                        "batch_id": path.name.replace("_output.jsonl", ""),
                        "custom_id": item.get("custom_id"),
                        "grammar_name": match.group("grammar_name"),
                        "input_hash": match.group("input_hash"),
                        "sample_id": match.group("sample_id"),
                        "model": body.get("model"),
                        "fuzzy_model": fuzzy_model(body.get("model")),
                        "model_response": message,
                        "model_answer": extract_answer(message),
                        "status_code": (item.get("response") or {}).get("status_code"),
                        "error": item.get("error"),
                        "prompt_tokens": prompt_tokens,
                        "completion_tokens": completion_tokens,
                        "total_tokens": total_tokens,
                    }
                )
    return pd.DataFrame(rows)


def load_inputs(batch_dir: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(batch_dir.glob("inputs_*.jsonl")):
        if path.name.endswith("_output.jsonl"):
            continue
        with open(path) as handle:
            for line in handle:
                item = json.loads(line)
                body = item.get("body") or {}
                metadata = body.get("metadata") or {}
                match = CUSTOM_ID_RE.match(item.get("custom_id", ""))
                rows.append(
                    {
                        "custom_id": item.get("custom_id"),
                        "input_file": path.name,
                        "fuzzy_model": fuzzy_model(body.get("model")),
                        "grammar_name": metadata.get("grammar_name")
                        or (match.group("grammar_name") if match else None),
                        "sample_id": (
                            str(metadata.get("sample_id"))
                            if metadata.get("sample_id") is not None
                            else (match.group("sample_id") if match else None)
                        ),
                        "depth": pd.to_numeric(metadata.get("depth"), errors="coerce"),
                    }
                )
    return pd.DataFrame(rows)


outputs_df = load_outputs(EXP_BATCH_DIR)
inputs_df = load_inputs(EXP_BATCH_DIR)
print(f"Output rows loaded: {len(outputs_df)}")
print(f"Input rows indexed: {len(inputs_df)}")

if len(outputs_df):
    display(
        outputs_df.groupby(["batch_file", "fuzzy_model"], observed=False)
        .size()
        .rename("n_rows")
        .reset_index()
    )
else:
    print("No batch output files found yet.")

## Merge outputs with samples and grammars

In [ ]:
merged_df = outputs_df.merge(
    inputs_df.drop_duplicates(subset=["custom_id", "fuzzy_model"]),
    on=["custom_id", "fuzzy_model"],
    how="left",
    suffixes=("", "_input"),
)

for column in ["grammar_name", "sample_id", "depth"]:
    fallback_column = f"{column}_input"
    if fallback_column in merged_df.columns:
        merged_df[column] = merged_df[column].combine_first(merged_df[fallback_column])
        merged_df = merged_df.drop(columns=[fallback_column])

if len(merged_df):
    merged_df = merged_df.drop_duplicates(subset=["batch_id", "custom_id"]).copy()
    merged_df = merged_df.merge(
        samples_df[
            [
                "grammar_name",
                "sample_id",
                "input_sentence",
                "output_sentence",
                "input_length",
            ]
        ],
        on=["grammar_name", "sample_id"],
        how="left",
        validate="many_to_one",
    )
    merged_df = merged_df.merge(
        grammars_df[
            [
                "grammar_name",
                "target_orthography",
                "target_orthography_key",
                "grammar_size",
                "n_rules",
                "n_words",
                "target_vocab",
                "syllable_structure",
            ]
        ],
        on="grammar_name",
        how="left",
        validate="many_to_one",
    )

    merged_df["answered"] = merged_df["model_answer"].notna()
    merged_df["target_script"] = merged_df["output_sentence"].map(detect_script)
    merged_df["answer_script"] = merged_df["model_answer"].map(detect_script)
    merged_df["script_match"] = merged_df["target_script"] == merged_df["answer_script"]

    try:
        merged_df["input_length_quintile"] = pd.qcut(
            merged_df["input_length"], q=5, duplicates="drop"
        )
        merged_df["input_length_quintile_mid"] = merged_df["input_length_quintile"].map(
            lambda interval: (interval.left + interval.right) / 2
            if pd.notna(interval)
            else np.nan
        )
    except ValueError:
        merged_df["input_length_quintile"] = pd.NA
        merged_df["input_length_quintile_mid"] = np.nan

    completion_snapshot = (
        merged_df.groupby(["fuzzy_model", "target_orthography"], observed=False)
        .agg(rows=("custom_id", "size"), answered=("answered", "sum"))
        .reset_index()
    )
    completion_snapshot["answer_rate"] = completion_snapshot[
        "answered"
    ] / completion_snapshot["rows"].replace(0, np.nan)
    display(completion_snapshot.sort_values(["fuzzy_model", "target_orthography"]))

    expected_per_model = len(samples_df)
    print(
        (
            merged_df.groupby("fuzzy_model", observed=False)["custom_id"]
            .nunique()
            .rename("completed_rows")
            .reset_index()
            .assign(
                expected_rows=expected_per_model,
                completion_rate=lambda df: df["completed_rows"] / df["expected_rows"],
            )
        )
    )
else:
    print("No outputs merged yet.")

merged_df.head() if len(merged_df) else merged_df

## Accuracy metrics

In [ ]:
try:
    import evaluate
except ImportError:
    evaluate = None

try:
    import sacrebleu
except ImportError:
    sacrebleu = None

if len(merged_df):

    def exact_match(row) -> bool:
        return (
            bool(row["model_answer"]) and row["model_answer"] == row["output_sentence"]
        )

    def bow_match(row) -> bool:
        return bag_equal(row["model_answer"], row["output_sentence"])

    def edit_similarity(row) -> float:
        return SequenceMatcher(
            None, row["model_answer"] or "", row["output_sentence"] or ""
        ).ratio()

    def bleu_score(row) -> float:
        if sacrebleu is None:
            return np.nan
        return (
            sacrebleu.sentence_bleu(
                row["model_answer"] or "", [row["output_sentence"] or ""]
            ).score
            / 100.0
        )

    def num_oov_words(row) -> int:
        if not row["model_answer"]:
            return 0
        return len(set(tokenize(row["model_answer"])) - set(row["target_vocab"] or []))

    merged_df["exact_match"] = merged_df.apply(exact_match, axis=1)
    merged_df["bow_match"] = merged_df.apply(bow_match, axis=1)
    merged_df["edit_similarity"] = merged_df.apply(edit_similarity, axis=1)
    merged_df["bleu"] = merged_df.apply(bleu_score, axis=1)
    merged_df["num_oov_words"] = merged_df.apply(num_oov_words, axis=1)

    if evaluate is not None:
        chrf = evaluate.load("chrf")
        merged_df["chrF++"] = [
            chrf.compute(
                predictions=[pred or ""],
                references=[[ref or ""]],
                beta=2,
                word_order=2,
            )["score"]
            / 100.0
            for pred, ref in zip(
                merged_df["model_answer"], merged_df["output_sentence"]
            )
        ]
    else:
        merged_df["chrF++"] = np.nan

    summary_by_model = (
        merged_df.groupby("fuzzy_model", observed=False)
        .agg(
            rows=("custom_id", "size"),
            answered=("answered", "sum"),
            exact_match=("exact_match", "mean"),
            bow_match=("bow_match", "mean"),
            script_match=("script_match", "mean"),
            edit_similarity=("edit_similarity", "mean"),
            bleu=("bleu", "mean"),
            chrf_pp=("chrF++", "mean"),
            mean_prompt_tokens=("prompt_tokens", "mean"),
            mean_completion_tokens=("completion_tokens", "mean"),
        )
        .reset_index()
    )
    for column in [
        "exact_match",
        "bow_match",
        "script_match",
        "edit_similarity",
        "bleu",
        "chrf_pp",
    ]:
        summary_by_model[column] = (100 * summary_by_model[column]).round(2)
    display(summary_by_model)

    by_condition = (
        merged_df.groupby(["target_orthography", "fuzzy_model"], observed=False)
        .agg(
            rows=("custom_id", "size"),
            answered=("answered", "sum"),
            exact_match=("exact_match", "mean"),
            bow_match=("bow_match", "mean"),
            script_match=("script_match", "mean"),
            edit_similarity=("edit_similarity", "mean"),
            bleu=("bleu", "mean"),
            chrf_pp=("chrF++", "mean"),
            mean_oov_words=("num_oov_words", "mean"),
        )
        .reset_index()
    )
    for column in [
        "exact_match",
        "bow_match",
        "script_match",
        "edit_similarity",
        "bleu",
        "chrf_pp",
    ]:
        by_condition[column] = (100 * by_condition[column]).round(2)
    display(by_condition.sort_values(["fuzzy_model", "target_orthography"]))
else:
    print("No outputs available for scoring yet.")

## Plots

In [ ]:
PLOT_MODEL = None
FULL_PLOT_MODEL = "google/gemma-3-4b-it"

In [ ]:
if len(merged_df):
    EXPERIMENT_NAME = "orthography_large"
    FULL_PERF_METRICS = [label for _, label in METRICS_FOR_PLOT]
    available_models = aes.ordered_models(merged_df["fuzzy_model"].dropna().unique())
    default_model = "gpt-5" if "gpt-5" in available_models else available_models[0]

    overview_selected_model = PLOT_MODEL or default_model
    print(f"Plotting overview model: {overview_selected_model}")

    plot_source_df = merged_df.loc[
        merged_df["fuzzy_model"] == overview_selected_model
    ].copy()
    plot_source_df["match_type"] = "Exact Match"
    plot_source_df["match_value"] = plot_source_df["exact_match"]

    plot_length_df = plot_source_df.copy()
    plot_size_df = plot_source_df.copy()

    full_selected_model = FULL_PLOT_MODEL or default_model
    print(f"Plotting full figure model: {full_selected_model}")

    full_plot_source_df = merged_df.loc[
        merged_df["fuzzy_model"] == full_selected_model,
        [
            "fuzzy_model",
            "grammar_size",
            "input_length_quintile_mid",
            "target_orthography",
            "exact_match",
            "bow_match",
            "bleu",
            "chrF++",
        ],
    ].copy()
    full_plot_source_df = full_plot_source_df.melt(
        id_vars=[
            "fuzzy_model",
            "grammar_size",
            "input_length_quintile_mid",
            "target_orthography",
        ],
        value_vars=[metric for metric, _ in METRICS_FOR_PLOT],
        var_name="match_type",
        value_name="match_value",
    )
    full_plot_source_df["match_type"] = full_plot_source_df["match_type"].replace(
        dict(METRICS_FOR_PLOT)
    )
    full_plot_length_df = full_plot_source_df.copy()
    full_plot_size_df = full_plot_source_df.copy()
else:
    plot_length_df = pd.DataFrame()
    plot_size_df = pd.DataFrame()
    full_plot_length_df = pd.DataFrame()
    full_plot_size_df = pd.DataFrame()
    overview_selected_model = None
    full_selected_model = None

In [ ]:
if len(full_plot_length_df):
    fig = plt.figure(
        figsize=(aes.COLM_PAPER_WIDTH_IN, aes.FIG_HEIGHT_DOUBLEROW_DIFFAXES_IN)
    )
    grid = fig.add_gridspec(2, len(FULL_PERF_METRICS), wspace=0.12, hspace=0.7)

    for r, split in enumerate(["grammar", "sentence"]):
        for i, pm in enumerate(FULL_PERF_METRICS):
            ax = fig.add_subplot(grid[r, i])
            metric_df = full_plot_source_df[full_plot_source_df["match_type"] == pm]

            if split == "grammar":
                sns.lineplot(
                    data=metric_df,
                    x="grammar_size",
                    y="match_value",
                    hue="target_orthography",
                    style="target_orthography",
                    hue_order=ORTHOGRAPHY_ORDER,
                    style_order=ORTHOGRAPHY_ORDER,
                    palette=PALETTE_ORTHOGRAPHY_LARGE,
                    markers=True,
                    dashes=False,
                    errorbar="ci",
                    ax=ax,
                )
                ax.set_title(pm, fontweight="normal", loc="center")
                ax.set_ylim(-0.05, 1.05)
                ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
                ax.yaxis.grid(True, linestyle="--", alpha=0.5)
                ax.xaxis.set_major_formatter(aes.KMB_FORMATTER)
                ax.set_xlim(0, 10_000)
                if i == 0:
                    ax.set_xlabel("Grammar size", loc="left")
                    ax.set_ylabel("Mean score")
                else:
                    ax.set_ylabel("")
                    ax.yaxis.set_ticklabels([])
                    ax.set_xlabel("")
                if i == len(FULL_PERF_METRICS) - 1:
                    ax.legend(
                        title="",
                        bbox_to_anchor=(1.05, 1.0),
                        loc="upper left",
                        borderaxespad=0,
                        frameon=False,
                    )
                else:
                    legend = ax.get_legend()
                    if legend is not None:
                        legend.remove()
            else:
                sns.lineplot(
                    data=metric_df,
                    x="input_length_quintile_mid",
                    y="match_value",
                    hue="target_orthography",
                    style="target_orthography",
                    hue_order=ORTHOGRAPHY_ORDER,
                    style_order=ORTHOGRAPHY_ORDER,
                    palette=PALETTE_ORTHOGRAPHY_LARGE,
                    markers=True,
                    dashes=False,
                    errorbar="ci",
                    ax=ax,
                )
                ax.set_title(None)
                ax.set_ylim(-0.05, 1.05)
                ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
                ax.yaxis.grid(True, linestyle="--", alpha=0.5)
                if i == 0:
                    ax.set_xlabel("Sentence length (binned into quintiles)", loc="left")
                    ax.set_ylabel("Mean score")
                else:
                    ax.set_ylabel("")
                    ax.yaxis.set_ticklabels([])
                    ax.set_xlabel("")
                legend = ax.get_legend()
                if legend is not None:
                    legend.remove()

    plt.subplots_adjust(left=0, bottom=0, right=1, top=1)
    aes.save_figure(
        FIGURES_DIR / f"{full_selected_model.replace('/', '_')}-{EXPERIMENT_NAME}-full",
        fig=fig,
    )
    plt.show()
else:
    print("No scored outputs available yet for full plotting.")

In [ ]:
if len(plot_length_df):
    fig = plt.figure(figsize=(aes.COLM_PAPER_WIDTH_IN, aes.FIG_HEIGHT_SINGLE_ROW_IN))
    grid = fig.add_gridspec(1, 2, wspace=0.15)
    axes = [fig.add_subplot(grid[0, 0]), fig.add_subplot(grid[0, 1])]

    with sns.plotting_context("paper", font_scale=1, rc=aes.rcs):
        ax = axes[0]
        sns.lineplot(
            data=plot_size_df,
            x="grammar_size",
            y="match_value",
            hue="target_orthography",
            style="target_orthography",
            hue_order=ORTHOGRAPHY_ORDER,
            style_order=ORTHOGRAPHY_ORDER,
            palette=PALETTE_ORTHOGRAPHY_LARGE,
            markers=True,
            markersize=8,
            dashes=False,
            errorbar="ci",
            ax=ax,
        )
        ax.set_xlabel("Grammar size")
        ax.set_ylabel("Exact match")
        ax.set_ylim(-0.05, 1.05)
        ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
        ax.yaxis.grid(True, linestyle="--", alpha=0.5)
        ax.xaxis.set_major_formatter(aes.KMB_FORMATTER)
        ax.set_xlim(0, 10_000)

        ax = axes[1]
        sns.lineplot(
            data=plot_length_df,
            x="input_length_quintile_mid",
            y="match_value",
            hue="target_orthography",
            style="target_orthography",
            hue_order=ORTHOGRAPHY_ORDER,
            style_order=ORTHOGRAPHY_ORDER,
            palette=PALETTE_ORTHOGRAPHY_LARGE,
            markers=True,
            markersize=8,
            dashes=False,
            errorbar="ci",
            ax=ax,
        )
        ax.set_xlabel("Sentence length (binned into 5 quantiles)")
        ax.set_ylabel("")
        ax.set_ylim(-0.05, 1.05)
        ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
        ax.yaxis.grid(True, linestyle="--", alpha=0.5)
        ax.yaxis.set_ticklabels([])

        left_legend = fig.axes[0].get_legend()
        if left_legend is not None:
            left_legend.remove()

        right_legend = fig.axes[1].get_legend()
        if right_legend is not None:
            right_legend.set_title("")
            sns.move_legend(
                fig.axes[1], "upper left", bbox_to_anchor=(1.02, 1.0), frameon=False
            )

    plt.subplots_adjust(left=0, bottom=0, right=1, top=1)
    aes.save_figure(
        FIGURES_DIR / f"{overview_selected_model}_orthography_large_overview", fig=fig
    )
    plt.show()
else:
    print("No scored outputs available yet for overview plotting.")

In [ ]:
EXPORT_DIR = NOTEBOOKS_DIR / "cache" / "combined-exp"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

if len(plot_size_df) and len(plot_length_df):
    export_size_df = plot_size_df.copy()
    export_size_df["experiment"] = "orthography_large"
    export_size_df["panel"] = "grammar_size"
    export_size_df["selected_model"] = overview_selected_model
    export_size_df.to_csv(
        EXPORT_DIR / "orthography_large_grammar_size.csv", index=False
    )

    export_length_df = plot_length_df.copy()
    export_length_df["experiment"] = "orthography_large"
    export_length_df["panel"] = "string_length"
    export_length_df["selected_model"] = overview_selected_model
    export_length_df.to_csv(
        EXPORT_DIR / "orthography_large_string_length.csv", index=False
    )

    print(f"Saved combined-exp inputs to {EXPORT_DIR}")
else:
    print("No orthography plot data available to export.")